# Chapter 7: The same function against a local model

This notebook runs the statements in `run-local-500.cypher`, one per cell, so you can see each result as you go. The text above each cell is the comment from the script.

- Run the cells one at a time, from the top.
- Before you start: Ollama is running and the model is pulled (`ollama pull tev1:0.8b`).
- Needs the 500 transactions from chapter 2 and the hosted Decision nodes from chapter 6.
- No key is used and nothing leaves your machine.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

You also need:

- The plugin JAR in the `plugins` folder of your DBMS, with `dbms.security.procedures.allowlist=jev.*` in `neo4j.conf`.
- For the hosted API only, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             pandas==2.2.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import pandas as pd

from IPython.display import display
from neo4j import GraphDatabase

## 3. Configuration

The connection details come from environment variables set before Jupyter was launched.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Helper

`run_cypher` runs one statement. A single map, such as a `jev.decide` result, prints as JSON. Anything else shows as a table.

In [5]:
def run_cypher(query):
    """Run one statement and show the result. A single map prints as JSON, anything
    else as a table. Statements that change the graph also report what changed."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [r.data() for r in result]
        summary = result.consume()

    if len(records) == 1 and len(records[0]) == 1 and isinstance(next(iter(records[0].values())), dict):
        print(json.dumps(next(iter(records[0].values())), indent=2, default=str))
    elif records:
        display(pd.DataFrame(records))
    else:
        print("No results.")

    changes = {name: getattr(summary.counters, name, 0) for name in (
        "nodes_created", "nodes_deleted", "relationships_created",
        "relationships_deleted", "properties_set", "constraints_added",
        "constraints_removed")}
    changes = {k: v for k, v in changes.items() if v}
    if changes:
        print("Changes:", ", ".join(f"{k} {v}" for k, v in changes.items()))
    ms = (summary.result_available_after or 0) + (summary.result_consumed_after or 0)
    print(f"Completed after {ms:,} ms")

## 6. The statements

### 1. One call to the local model. authenticated is false: the key is only ever sent to Jev.

In [6]:
run_cypher("""
MATCH (t:Transaction {txn_id: 'T0001'})
RETURN jev.decide(
  t {.amount, .merchant_category, .distance_from_home_km, .hour_of_day,
     .account_age_days, .prior_flags, .txns_last_24h, .typical_monthly_spend},
  {decision: {
     type: 'choice',
     instructions: 'Based on all available signals, should this transaction be flagged for review or passed through?',
     criteria: {
       Flag: 'Potential fraud. Send for review.',
       Pass: 'Likely legitimate. Process normally.'
     }
  }},
  {url: 'http://localhost:11434/v1/systemone', model: 'tev1:0.8b'}
) AS result
""")

{
  "answers": {
    "decision": {
      "type": "choice",
      "choice": "Pass",
      "probabilities": {
        "Pass": 0.7310585786300049,
        "Flag": 0.2689414213699951
      },
      "confidence": 0.16005846201683083
    }
  },
  "error_message": null,
  "metadata": {
    "authenticated": false,
    "model": "tev1:0.8b",
    "latency_ms": 961,
    "attempts": 1
  }
}
Completed after 972 ms


### 2. Decide every transaction locally, 4 calls at a time, and store each answer as a LocalDecision node. The hosted Decision nodes from chapter 6 are left alone. margin is the gap between the two probabilities, so local and hosted confidence can be compared on one scale. Neo4j shows how long the statement took.

In [7]:
run_cypher("""
MATCH (t:Transaction)
WHERE NOT (t)-[:HAS_LOCAL_DECISION]->()
WITH collect({id: t.txn_id,
              state: t {.amount, .merchant_category, .distance_from_home_km, .hour_of_day,
                        .account_age_days, .prior_flags, .txns_last_24h, .typical_monthly_spend}}) AS items
CALL jev.decideAll(
  items,
  {decision: {
     type: 'choice',
     instructions: 'Based on all available signals, should this transaction be flagged for review or passed through?',
     criteria: {
       Flag: 'Potential fraud. Send for review.',
       Pass: 'Likely legitimate. Process normally.'
     }
  }},
  {url: 'http://localhost:11434/v1/systemone', model: 'tev1:0.8b', concurrency: 4}
) YIELD id, answers, error_message, metadata
WITH id, answers, metadata, error_message
WHERE error_message IS NULL
MATCH (t:Transaction {txn_id: id})
MERGE (t)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
SET l.choice     = answers.decision.choice,
    l.confidence = answers.decision.confidence,
    l.p_flag     = answers.decision.probabilities.Flag,
    l.p_pass     = answers.decision.probabilities.Pass,
    l.margin     = abs(answers.decision.probabilities.Flag - answers.decision.probabilities.Pass),
    l.latency_ms = metadata.latency_ms,
    l.attempts   = metadata.attempts,
    l.model      = metadata.model
RETURN count(l) AS local_decisions_stored
""")

,local_decisions_stored
0,500


Changes: nodes_created 500, relationships_created 500, properties_set 4000
Completed after 44,843 ms


### 3. How many transactions have a local decision? Expect 500. If some calls failed, run statement 2 again: it only picks up transactions without one.

In [8]:
run_cypher("""
MATCH (t:Transaction)
OPTIONAL MATCH (t)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
RETURN count(DISTINCT t) AS transactions, count(l) AS with_local_decision
""")

,transactions,with_local_decision
0,500,500


Completed after 20 ms


### 4. The local split: how many were flagged and how many passed?

In [9]:
run_cypher("""
MATCH (:Transaction)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
RETURN l.choice AS choice, count(*) AS n
ORDER BY n DESC
""")

,choice,n
0,Pass,480
1,Flag,20


Completed after 6 ms


### 5. Local decisions against the hidden label. is_fraud was never sent to the model.

In [10]:
run_cypher("""
MATCH (t:Transaction)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
RETURN l.choice AS choice, t.is_fraud AS is_fraud, count(*) AS n
ORDER BY choice, is_fraud
""")

,choice,is_fraud,n
0,Flag,0,4
1,Flag,1,16
2,Pass,0,307
3,Pass,1,173


Completed after 6 ms


### 6. Hosted against local: how often do the two agree?

In [11]:
run_cypher("""
MATCH (t:Transaction)-[:HAS_DECISION]->(h:Decision),
      (t)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
RETURN h.choice AS hosted, l.choice AS local, count(*) AS n
ORDER BY hosted, local
""")

,hosted,local,n
0,Flag,Flag,20
1,Flag,Pass,248
2,Pass,Pass,232


Completed after 5 ms


### 7. Where they disagree, by scenario.

In [12]:
run_cypher("""
MATCH (t:Transaction)-[:HAS_DECISION]->(h:Decision),
      (t)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
WHERE h.choice <> l.choice
RETURN t.scenario AS scenario, count(*) AS disagreements
ORDER BY disagreements DESC
""")

,scenario,disagreements
0,card_testing,64
1,clear_fraud,54
2,night_atm,51
3,traveler,47
4,takeover,31
5,big_purchase,1


Completed after 8 ms


### 8. Confidence on one scale. The reported confidence means different things (the margin for the hosted service, one minus the normalized entropy locally). The margin computed from the probabilities is comparable.

In [13]:
run_cypher("""
MATCH (t:Transaction)-[:HAS_DECISION]->(h:Decision),
      (t)-[:HAS_LOCAL_DECISION]->(l:LocalDecision)
RETURN round(avg(h.confidence), 3)                  AS hosted_reported_confidence,
       round(avg(abs(h.p_flag - h.p_pass)), 3)      AS hosted_margin,
       round(avg(l.confidence), 3)                  AS local_reported_confidence,
       round(avg(l.margin), 3)                      AS local_margin
""")

,hosted_reported_confidence,hosted_margin,local_reported_confidence,local_margin
0,0.693,0.693,0.122,0.356


Completed after 16 ms


### 9. Latency per call, as measured inside the function.

In [14]:
run_cypher("""
MATCH (h:Decision) WITH avg(h.latency_ms) AS hosted_ms
MATCH (l:LocalDecision)
RETURN round(hosted_ms, 1) AS hosted_avg_ms, round(avg(l.latency_ms), 1) AS local_avg_ms
""")

,hosted_avg_ms,local_avg_ms
0,243.1,348.7


Completed after 5 ms


### 10. Reset (optional, leave the switch set to False). To run the local batch again from scratch, delete only the local decisions. The hosted decisions and the transactions stay.

In [15]:
RESET_LOCAL_DECISIONS = False   # True: delete the LocalDecision nodes

if RESET_LOCAL_DECISIONS:
    run_cypher("""MATCH (l:LocalDecision) DETACH DELETE l""")
else:
    print("Skipped. Set RESET_LOCAL_DECISIONS = True to delete the LocalDecision nodes.")

Skipped. Set RESET_LOCAL_DECISIONS = True to delete the LocalDecision nodes.


## 7. Teardown

In [16]:
driver.close()
print("Connection closed.")

Connection closed.
